# Notebook 1 — Read & Join the Tables

**Goal:** read every Olist table on its own, understand what one row means in each,
aggregate the tables that have multiple rows per order, then join everything into
a single **ML table with one row per order**.

**Artifact produced by this notebook:** `artifacts/ml_table_orders.parquet`


In [ ]:


import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from pathlib import Path

pd.set_option("display.max_columns", None)

In [24]:
DB_USER = "ahmed"
DB_PASSWORD = ""
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "qafza"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version();")).fetchone()[0])

PostgreSQL 15.19 (Homebrew) on aarch64-apple-darwin25.6.0, compiled by Apple clang version 21.0.0 (clang-2100.1.1.101), 64-bit


In [ ]:
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

## 1. Read every table on its own

For each table: shape, dtypes, a preview, duplicate check on the expected key,
and a one-line note on what a single row represents. This is what tells us
which tables need aggregating before we can join them.

In [26]:
tables = pd.read_sql(
    """SELECT table_name FROM information_schema.tables
       WHERE table_schema = 'public' ORDER BY table_name;""",
    engine,
)["table_name"].tolist()
tables

['customers',
 'geolocation',
 'order_items',
 'order_payments',
 'order_reviews',
 'orders',
 'product_category_name_translation',
 'products',
 'sellers']

In [27]:
dfs = {t: pd.read_sql(f"SELECT * FROM {t};", engine) for t in tables}

for name, df in dfs.items():
    print(f"{name:25s} shape={df.shape}")

customers                 shape=(99441, 5)
geolocation               shape=(1000163, 5)
order_items               shape=(112650, 7)
order_payments            shape=(103886, 5)
order_reviews             shape=(99224, 7)
orders                    shape=(99441, 8)
product_category_name_translation shape=(71, 2)
products                  shape=(32951, 9)
sellers                   shape=(3095, 4)


### customers 

In [28]:
df = dfs["customers"]
print("duplicate customer_id rows:", df["customer_id"].duplicated().sum())
print("unique customer_unique_id:", df["customer_unique_id"].nunique(), "vs rows:", len(df))
df.head()

duplicate customer_id rows: 0
unique customer_unique_id: 96096 vs rows: 99441


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


customer_id is unique per row, but customer_unique_id repeats. Olist gives a
returning customer a new customer_id for each order but keeps the same
customer_unique_id across orders. Since we're building a table at the order
level, customer_id is the key we want here.

### orders 

In [29]:
df = dfs["orders"]
print("duplicate order_id rows:", df["order_id"].duplicated().sum())
print(df["order_status"].value_counts())
df.head()

duplicate order_id rows: 0
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


### order_items 

In [30]:
df = dfs["order_items"]
print("rows:", len(df), " unique order_id:", df["order_id"].nunique())
print("avg items per order:", round(len(df) / df["order_id"].nunique(), 2))
df.head()

rows: 112650  unique order_id: 98666
avg items per order: 1.14


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


This one needs to be aggregated before joining, otherwise merging it straight onto orders would duplicate order rows (one row per item instead of one per order).

### order_payments

In [31]:
df = dfs["order_payments"]
print("rows:", len(df), " unique order_id:", df["order_id"].nunique())
print("avg payment rows per order:", round(len(df) / df["order_id"].nunique(), 2))
df["payment_type"].value_counts()

rows: 103886  unique order_id: 99440
avg payment rows per order: 1.04


payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Same story as order_items, needs to be aggregated first.

### order_reviews

In [32]:
df = dfs["order_reviews"]
print("rows:", len(df), " unique order_id:", df["order_id"].nunique())
dup_orders = df["order_id"].value_counts()
print("orders with >1 review row:", (dup_orders > 1).sum())
df.head()

rows: 99224  unique order_id: 98673
orders with >1 review row: 547


,review_id,order_id,review_score,review_comment_tittle,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53


A small number of orders have more than one review, so aggregating this one too (average score, count of reviews).

### products, sellers, product_category_name_translation, geolocation

In [33]:
for t in ["products", "sellers", "product_category_name_translation", "geolocation"]:
    df = dfs[t]
    print(f"\n--- {t} ---  shape={df.shape}")
    key_col = {"products": "product_id", "sellers": "seller_id",
               "product_category_name_translation": "product_category_name"}.get(t)
    if key_col:
        print("duplicate", key_col, ":", df[key_col].duplicated().sum())
    display(df.head(3))


--- products ---  shape=(32951, 9)
duplicate product_id : 0


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_lenght_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0



--- sellers ---  shape=(3095, 4)
duplicate seller_id : 0


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ



--- product_category_name_translation ---  shape=(71, 2)
duplicate product_category_name : 0


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto



--- geolocation ---  shape=(1000163, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP


geolocation has no unique key by design, many lat/lng rows per zip prefix. Leaving it out of the order-level table for now, can join it in later through customer_zip_code_prefix / seller_zip_code_prefix if geography features are needed.

## 2. Aggregate before you join

`order_items`, `order_payments`, and `order_reviews` all have multiple rows per order.
We collapse each to exactly one row per `order_id` before joining anything onto `orders`.

In [34]:
# order_items -> one row per order
items = dfs["order_items"]

items_agg = items.groupby("order_id").agg(
    num_items=("order_item_id", "count"),
    num_distinct_products=("product_id", "nunique"),
    num_distinct_sellers=("seller_id", "nunique"),
    total_price=("price", "sum"),
    total_freight_value=("freight_value", "sum"),
    avg_item_price=("price", "mean"),
).reset_index()

print(items_agg.shape)
items_agg.head()

(98666, 7)


,order_id,num_items,num_distinct_products,num_distinct_sellers,total_price,total_freight_value,avg_item_price
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29,58.90
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93,239.90
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87,199.00
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14,199.90


In [35]:
# order_payments -> one row per order
payments = dfs["order_payments"]

payments_agg = payments.groupby("order_id").agg(
    num_payments=("payment_sequential", "count"),
    total_payment_value=("payment_value", "sum"),
    max_installments=("payment_installments", "max"),
).reset_index()

# most common payment type per order (mode)
payment_type_mode = (
    payments.groupby("order_id")["payment_type"]
    .agg(lambda s: s.mode().iat[0] if not s.mode().empty else None)
    .reset_index()
    .rename(columns={"payment_type": "main_payment_type"})
)

payments_agg = payments_agg.merge(payment_type_mode, on="order_id", how="left")

print(payments_agg.shape)
payments_agg.head()

(99440, 5)


,order_id,num_payments,total_payment_value,max_installments,main_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,1,216.87,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,3,credit_card


In [36]:
# order_reviews -> one row per order
reviews = dfs["order_reviews"]

reviews_agg = reviews.groupby("order_id").agg(
    avg_review_score=("review_score", "mean"),
    num_reviews=("review_id", "count"),
).reset_index()

print(reviews_agg.shape)
reviews_agg.head()

(98673, 3)


,order_id,avg_review_score,num_reviews
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


## 3. Join into one ML table, one row per order

Start from `orders` and left-join everything else onto it.


In [37]:
orders = dfs["orders"]
customers = dfs["customers"]

ml_table = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
)

print("orders rows:      ", len(orders))
print("ml_table rows:     ", len(ml_table))
print("unique order_id:  ", ml_table["order_id"].nunique())

orders rows:       99441
ml_table rows:      99441
unique order_id:   99441


In [38]:
# the join should not have duplicated any order rows, checking just in case
assert len(ml_table) == len(orders), "row count changed during join, a duplicate key slipped in somewhere"
assert ml_table["order_id"].is_unique, "order_id is not unique in the final table"
print("one row per order, looks good")

ml_table.head()

one row per order, looks good


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,num_items,num_distinct_products,num_distinct_sellers,total_price,total_freight_value,avg_item_price,num_payments,total_payment_value,max_installments,main_payment_type,avg_review_score,num_reviews
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,03149,sao paulo,SP,1.0,1.0,1.0,29.99,8.72,29.99,3.0,38.71,1.0,voucher,4.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,1.0,1.0,118.70,22.76,118.70,1.0,141.46,1.0,boleto,4.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,1.0,1.0,159.90,19.22,159.90,1.0,179.12,3.0,credit_card,5.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,1.0,1.0,45.00,27.20,45.00,1.0,72.20,1.0,credit_card,5.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,09195,santo andre,SP,1.0,1.0,1.0,19.90,8.72,19.90,1.0,28.62,1.0,credit_card,5.0,1.0


In [39]:
ml_table.isna().sum().sort_values(ascending=False).head(10)

order_delivered_customer_date    2965
order_delivered_carrier_date     1783
num_items                         775
num_distinct_products             775
avg_item_price                    775
total_freight_value               775
total_price                       775
num_distinct_sellers              775
avg_review_score                  768
num_reviews                       768
dtype: int64

## 4. Adding seller state, distance, and product weight

Coming back to this after the first model in notebook 6 came out weak (PR-AUC
close to what a no-skill model would get). Nothing correlated much with is_late
using what was in the table before, which pointed at missing the two things
that usually actually drive delivery delays: how far the package has to
travel, and how heavy/bulky it is. Neither was in the ML table, geolocation
and product dimensions got left out earlier since they needed extra joins.
Adding them now.

geolocation doesn't have a clean key (many lat/lng rows per zip prefix), so
building a lookup table first, one average lat/lng per zip prefix.

In [40]:
geo = dfs["geolocation"]

geo_lookup = (
    geo.groupby("geolocation_zip_code_prefix")
    .agg(lat=("geolocation_lat", "mean"), lng=("geolocation_lng", "mean"))
    .reset_index()
    .rename(columns={"geolocation_zip_code_prefix": "zip_code_prefix"})
)

print(geo_lookup.shape)
geo_lookup.head()

(19015, 3)


,zip_code_prefix,lat,lng
0,01001,-23.550190,-46.634024
1,01002,-23.548146,-46.634979
2,01003,-23.548994,-46.635731
3,01004,-23.549799,-46.634757
4,01005,-23.549456,-46.636733


In [42]:
# pull in seller state + zip, and product weight/dimensions at the item level,
# then compute the distance for each item using the zip lookup above

items_detail = dfs["order_items"].merge(
    dfs["sellers"][["seller_id", "seller_zip_code_prefix", "seller_state"]],
    on="seller_id", how="left",
).merge(
    dfs["products"][["product_id", "product_weight_g", "product_lenght_cm",
                       "product_height_cm", "product_width_cm"]],
    on="product_id", how="left",
).merge(
    dfs["customers"][["customer_id", "customer_zip_code_prefix"]],
    left_on="order_id", right_on="customer_id", how="left",  # placeholder, fixed below
)

items_detail.shape

(112650, 15)

In [44]:
order_to_customer = orders[["order_id", "customer_id"]].merge(
    customers[["customer_id", "customer_zip_code_prefix"]], on="customer_id", how="left"
)

items_detail = dfs["order_items"].merge(
    dfs["sellers"][["seller_id", "seller_zip_code_prefix", "seller_state"]],
    on="seller_id", how="left",
).merge(
    dfs["products"][["product_id", "product_weight_g", "product_lenght_cm",
                       "product_height_cm", "product_width_cm"]],
    on="product_id", how="left",
).merge(
    order_to_customer, on="order_id", how="left",
)

items_detail.shape

(112650, 15)

In [45]:
items_detail = items_detail.merge(
    geo_lookup.rename(columns={"lat": "seller_lat", "lng": "seller_lng"}),
    left_on="seller_zip_code_prefix", right_on="zip_code_prefix", how="left",
).merge(
    geo_lookup.rename(columns={"lat": "customer_lat", "lng": "customer_lng"}),
    left_on="customer_zip_code_prefix", right_on="zip_code_prefix", how="left",
    suffixes=("", "_cust"),
)

print("rows missing seller coords:", items_detail["seller_lat"].isna().sum())
print("rows missing customer coords:", items_detail["customer_lat"].isna().sum())

rows missing seller coords: 253
rows missing customer coords: 302


In [46]:
def haversine_km(lat1, lng1, lat2, lng2):
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))

items_detail["distance_km"] = haversine_km(
    items_detail["seller_lat"], items_detail["seller_lng"],
    items_detail["customer_lat"], items_detail["customer_lng"],
)

items_detail["distance_km"].describe()

count    112096.000000
mean        596.959183
std         589.978820
min           0.000000
25%         184.065837
50%         431.635462
75%         792.275606
max        8677.911622
Name: distance_km, dtype: float64

In [48]:
geo_product_agg = items_detail.groupby("order_id").agg(
    avg_distance_km=("distance_km", "mean"),
    total_product_weight_g=("product_weight_g", "sum"),
    avg_product_weight_g=("product_weight_g", "mean"),
    avg_product_length_cm=("product_lenght_cm", "mean"),
    avg_product_height_cm=("product_height_cm", "mean"),
    avg_product_width_cm=("product_width_cm", "mean"),
).reset_index()

main_seller_state = (
    items_detail.groupby("order_id")["seller_state"]
    .agg(lambda s: s.mode().iat[0] if not s.mode().empty else None)
    .reset_index()
    .rename(columns={"seller_state": "main_seller_state"})
)

geo_product_agg = geo_product_agg.merge(main_seller_state, on="order_id", how="left")

print(geo_product_agg.shape)
geo_product_agg.head()

(98666, 8)


,order_id,avg_distance_km,total_product_weight_g,avg_product_weight_g,avg_product_length_cm,avg_product_height_cm,avg_product_width_cm,main_seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,301.504681,650.0,650.0,28.0,9.0,14.0,SP
1,00018f77f2f0320c557190d7a144bdd3,585.563937,30000.0,30000.0,50.0,30.0,40.0,SP
2,000229ec398224ef6ca0657da4fc703e,312.343511,3050.0,3050.0,33.0,13.0,33.0,MG
3,00024acbcdf0a6daa1e931b038114c75,293.168420,200.0,200.0,16.0,10.0,15.0,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,646.163463,3750.0,3750.0,35.0,40.0,30.0,PR


In [49]:
ml_table = ml_table.merge(geo_product_agg, on="order_id", how="left")

assert len(ml_table) == len(orders), "row count changed after adding geo/product features"
print("still one row per order:", ml_table["order_id"].is_unique)
print("missing distance_km:", ml_table["avg_distance_km"].isna().sum(),
      "out of", len(ml_table))

still one row per order: True
missing distance_km: 1264 out of 99441


Some rows will end up missing distance or weight, mainly when a zip prefix doesn't show up in the geolocation table at all, that gets handled with imputation in notebook 5, not here, this notebook's job is just building the raw features.

## 5. Save the artifact

This is the file Notebook 2 (labeling) will read — it should never need to query
the database again.

In [50]:
output_path = ARTIFACTS_DIR / "ml_table_orders.parquet"
ml_table.to_parquet(output_path, index=False)

print(f"Saved: {output_path}  ({ml_table.shape[0]} rows, {ml_table.shape[1]} columns)")

Saved: artifacts/ml_table_orders.parquet  (99441 rows, 31 columns)


Went through each table on its own first to see what a row actually means there,
then aggregated order_items and order_payments since those have several rows per
order. Joined everything onto orders and checked the row count didn't change after
the merge, so nothing got duplicated.

Came back later and added seller state, distance between seller and customer,
and product weight/dimensions, once notebook 6's first model made it clear
the original features weren't carrying much signal on their own.

